In [ ]:
from transformers import AutoTokenizer, AutoModel
from torch import nn
import torch
import os
import numpy as np
import random
import math

seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
from types import SimpleNamespace


Arguments = SimpleNamespace

args = Arguments(
    train_data='data/multitask-data/merged_3_data_5k_each.csv',
    batch_size=32,
    val_batch_size=64,

    query_max_len=256,
    passage_max_len=256,

    temperature=0.1,

    student_device='cuda:0',
    num_train_epochs=30,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,

    output_dir='ckpt/bert/bert-checkpoint',
    student_model='bert-base-uncased',
    student_tokenizer='bert-base-uncased',
)


teacher_embedding_path = 'data/embedding-multitask-data/qwen3-4B-embedding-multi-data.pt'

In [ ]:
def get_embedding(hidden_state, attention_mask):
    cls_token = hidden_state[:, 0:1, :]

    attention_scores = torch.bmm(cls_token, hidden_state.transpose(1, 2))
    attention_scores = attention_scores.squeeze(1)

    hidden_dim = hidden_state.size(-1)
    attention_scores = attention_scores / math.sqrt(hidden_dim)

    extended_attention_mask = (1.0 - attention_mask) * -1e9
    # extended_attention_mask[: 0] = -1e9
    attention_scores = attention_scores + extended_attention_mask
    weights = F.softmax(attention_scores, dim=-1)
    weights_expanded = weights.unsqueeze(-1)  # (batch_size, seq_len, 1)

    pooled_vector = torch.sum(hidden_state * weights_expanded, dim=1)

    return pooled_vector

In [ ]:
from torch.nn.functional import pad
from torch.utils.data import Dataset
import pandas as pd
import torch
from transformers import PreTrainedTokenizer
from dataclasses import dataclass
import json

from torch.utils.data import DataLoader, Dataset

class BiEncoderDataset(Dataset):
    def __init__(self, file_path, teacher_embedding_path):
        self.dataset = pd.read_csv(file_path)
        self.teacher_embedding = torch.load(teacher_embedding_path)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        query = self.dataset.iloc[index]['text']
        teacher_embedding = self.teacher_embedding[index]
        return query, teacher_embedding

@dataclass
class BiDataCollator:
    student_tokenizer: PreTrainedTokenizer = None
    max_len: int = 64
    return_tensors: str = 'pt'
    padding: bool = True


    def __call__(self, batch):
        queries = []
        teacher_embeddings = []
        for q, t_embed in batch:
            queries.append(q)
            teacher_embeddings.append(t_embed)

        teacher_embeddings = torch.stack(teacher_embeddings)

        student_inputs = self.student_tokenizer(
            queries,
            truncation=True,
            padding=self.padding,
            max_length=self.max_len,
            return_tensors=self.return_tensors,
        )

        return (student_inputs, teacher_embeddings)


In [ ]:
student_tokenizer = AutoTokenizer.from_pretrained(args.student_tokenizer)

train_dataset = BiEncoderDataset(args.train_data, teacher_embedding_path)

train_collate = BiDataCollator(student_tokenizer,
                               max_len = args.passage_max_len,
                               return_tensors = 'pt', padding = True)

train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                          shuffle=True, collate_fn=train_collate)

args.teacher_embedding_dimension = train_dataset.teacher_embedding.size(-1)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr

tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

class STSDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # instruction = "Given a text, Retrieve semantically similar text: "
        instruction=""
        return {
            "sentence1": instruction + self.dataset.iloc[idx]['sentence1'],
            "sentence2": instruction + self.dataset.iloc[idx]['sentence2'],
            "label": torch.tensor(self.dataset.iloc[idx]['score'], dtype=torch.float),
        }

def collate_fn(batch, tokenizer, max_len=256):
    s1_list = [item["sentence1"] for item in batch]
    s2_list = [item["sentence2"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])

    enc1 = tokenizer(
        s1_list,
        truncation=True,
        padding=True,       # chỉ pad theo câu dài nhất trong batch
        max_length=max_len,
        return_tensors="pt"
    )
    enc2 = tokenizer(
        s2_list,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors="pt"
    )

    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "input_ids2": enc2["input_ids"],
        "attention_mask2": enc2["attention_mask"],
        "labels": labels,
    }

def eval_sts(model, eval_loader):
    preds, labels = [], []
    device = model.device

    with torch.cuda.amp.autocast(dtype=torch.float16):
        with torch.no_grad():
            for batch in tqdm(eval_loader):
                input_ids1 = batch["input_ids1"].to(device)
                attn1 = batch["attention_mask1"].to(device)
                input_ids2 = batch["input_ids2"].to(device)
                attn2 = batch["attention_mask2"].to(device)
                label = batch["labels"]


                out1 = model(input_ids=input_ids1, attention_mask=attn1)
                out2 = model(input_ids=input_ids2, attention_mask=attn2)

                emb1 = get_embedding(out1.last_hidden_state, attn1)
                emb2 = get_embedding(out2.last_hidden_state, attn2)

                # cosine similarity
                sim = F.cosine_similarity(emb1, emb2)
                score = (sim + 1) * 2.5  # scale [-1,1] -> [0,5]

                preds.extend(score.cpu().numpy())
                labels.extend(label.numpy())

    spearman_corr, _ = spearmanr(preds, labels)
    print(f"Spearman: {spearman_corr:.4f}")

    return spearman_corr


def eval_sts_task(model, path_list):
    model.eval()
    print('eval_sts_task')
    sum_spearman_corr = 0
    for path in path_list:
        print(path)
        eval_dataset = STSDataset(path)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=128,
            shuffle=False,
            collate_fn=lambda x: collate_fn(x, tokenizer)
        )
        spearman_corr = eval_sts(model, eval_loader)
        sum_spearman_corr += spearman_corr

    model.train()

    return sum_spearman_corr / len(path_list)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
import datasets
import numpy as np
import torch

def eval_cls(model, eval_loader):
    preds, labels = [], []
    device = model.device

    with torch.cuda.amp.autocast(dtype=torch.float16):
        with torch.no_grad():
            for batch in tqdm(eval_loader):
                input_ids1 = batch["input_ids1"].to(device)
                attn1 = batch["attention_mask1"].to(device)
                label = batch["labels"]

                out1 = model(input_ids=input_ids1, attention_mask=attn1)
                emb1 = get_embedding(out1.last_hidden_state, attn1)

                preds.extend(emb1.cpu().numpy())
                labels.extend(label.numpy())

    return preds, labels

class ClasssifyDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return {
            "text": self.dataset.iloc[idx]['text'],
            "label": torch.tensor(self.dataset.iloc[idx]['label'], dtype=torch.long),
        }

def clf_collate_fn(batch, tokenizer, max_len=256):
    s1_list = [item["text"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])

    enc1 = tokenizer(
        s1_list,
        truncation=True,
        padding=True,       # chỉ pad theo câu dài nhất trong batch
        max_length=max_len,
        return_tensors="pt"
    )

    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "labels": labels,
    }


def eval_classification_task(model, path_list):
    model.eval()
    print('classifier')
    sum_f1 = 0

    for train_path, dev_path in path_list:
        print(dev_path)
        eval_dataset = ClasssifyDataset(dev_path)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=128,
            shuffle=False,
            collate_fn=lambda x: clf_collate_fn(x, tokenizer)
        )

        train_dataset = ClasssifyDataset(train_path)
        train_loader = DataLoader(
            train_dataset,
            batch_size=64,
            shuffle=False,
            collate_fn=lambda x: clf_collate_fn(x, tokenizer)
        )

        X_train, y_train = eval_cls(model, train_loader)
        X_test, y_test = eval_cls(model, eval_loader)

        clf = LogisticRegression(
            random_state=42,
            n_jobs=1,
            max_iter=200,
            verbose=0,
        )
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        scores = {}
        accuracy = accuracy_score(y_test, y_pred)
        scores["accuracy"] = accuracy
        f1 = f1_score(y_test, y_pred, average="macro")
        scores["f1"] = f1
        print(scores)
        sum_f1 += f1

    model.train()

    return sum_f1 / len(path_list)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, average_precision_score


tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

class PairDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # instruction = "Given a text, Retrieve semantically similar text: "
        instruction=""
        return {
            "sentence1": instruction + self.dataset.iloc[idx]['sentence1'],
            "sentence2": instruction + self.dataset.iloc[idx]['sentence2'],
            "label": torch.tensor(self.dataset.iloc[idx]['label'], dtype=torch.float),
        }


def eval_pair(model, eval_loader):
    preds, labels = [], []
    device = model.device

    with torch.cuda.amp.autocast(dtype=torch.float16):
        with torch.no_grad():
            for batch in tqdm(eval_loader):
                input_ids1 = batch["input_ids1"].to(device)
                attn1 = batch["attention_mask1"].to(device)
                input_ids2 = batch["input_ids2"].to(device)
                attn2 = batch["attention_mask2"].to(device)
                label = batch["labels"]


                out1 = model(input_ids=input_ids1, attention_mask=attn1)
                out2 = model(input_ids=input_ids2, attention_mask=attn2)

                emb1 = get_embedding(out1.last_hidden_state, attn1)
                emb2 = get_embedding(out2.last_hidden_state, attn2)

                # cosine similarity
                sim = F.cosine_similarity(emb1, emb2)
                score = (sim + 1) / 2

                preds.extend(score.cpu().numpy())
                labels.extend(label.numpy())

    metric = get_metric_pair_classification(preds, labels)
    print(metric)

    return metric

def get_metric_pair_classification(scores, labels):
    best_acc, best_thr = 0, 0
    for thr in np.linspace(0, 1, 200):
        preds = (scores >= thr).astype(int)
        acc = accuracy_score(labels, preds)
        if acc > best_acc:
            best_acc, best_thr = acc, thr
    preds = (scores >= best_thr).astype(int)
    return {
        "best_threshold": best_thr,
        "accuracy": best_acc,
        "f1": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="macro"),
        "recall": recall_score(labels, preds, average="macro"),
        "average_precision": average_precision_score(labels, scores)
    }


def eval_pair_task(model, path_list):
    model.eval()
    print('eval_pair_task')
    sum_average_precision = 0
    for path in path_list:
        print(path)
        eval_dataset = PairDataset(path)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=128,
            shuffle=False,
            collate_fn=lambda x: collate_fn(x, tokenizer)
        )
        metric = eval_pair(model, eval_loader)
        sum_average_precision += metric['average_precision']

    model.train()
    return sum_average_precision / len(path_list)



In [ ]:


class Trainer:
    def __init__(self, student, projectors, args):
        super().__init__()

        self.student = student.train()
        self.projector = projectors
        self.device = student.device
        self.n_layer = student.config.num_hidden_layers

        self.cross_entropy = nn.CrossEntropyLoss(reduction='mean')
        self.mse_loss = nn.MSELoss(reduction='mean')

        self.args = args

        self.step = 0

    def compute_scores(self, q_reps, p_reps, temperature=1.0):
        scores = torch.matmul(q_reps, p_reps.transpose(0, 1)) / temperature
        scores = scores.view(q_reps.size(0), -1)
        return scores

    def std_norm(self, embeddings):
        std_embeddings = embeddings.std(dim=-1, keepdim=True).detach()
        norm_embeddings = embeddings / std_embeddings
        return norm_embeddings

    def mse_dim_weight_loss(self, student_embeddings, teacher_embeddings, power=1.0):
        std_embeddings = teacher_embeddings.std(dim=(0))
        mean_std = std_embeddings.mean()
        std_scaled = std_embeddings / mean_std
        weights = std_scaled ** power

        squared_diff = (student_embeddings - teacher_embeddings) ** 2
        weighted_squared_diff = squared_diff * weights
        weighted_mse_loss = weighted_squared_diff.mean()

        return weighted_mse_loss

    def structure_loss(self, student_embeddings, teacher_embeddings):
        student_embeddings = F.normalize(student_embeddings, p=2, dim=-1)
        teacher_embeddings = F.normalize(teacher_embeddings, p=2, dim=-1)

        student_similarity = student_embeddings @ student_embeddings.transpose(-1, -2)
        teacher_similarity = teacher_embeddings @ teacher_embeddings.transpose(-1, -2)

        loss = F.mse_loss(student_similarity, teacher_similarity)

        return loss

    def cosine_loss(self, student_embeddings, teacher_embeddings):
        cos_sim = F.cosine_similarity(student_embeddings, teacher_embeddings, dim=-1)
        cos_sim_loss = 1 - cos_sim
        return cos_sim_loss.mean()

    def distill_loss(self, student_outputs, teacher_embedings=None, attention_mask=None):
        hidden_states = student_outputs.hidden_states

        if teacher_embedings is not None:
            if student_outputs.hidden_states is not None:
                TAMD_loss, LASD_loss = 0, 0

                n_layer_distill = 2
                for i in range(self.n_layer - n_layer_distill + 1, self.n_layer + 1):
                    h_embed = self.get_embedding(hidden_states[i], attention_mask)
                    h_proj = self.projector[i - 1](h_embed)
                    TAMD_loss += self.cosine_loss(h_proj, teacher_embedings)

                    # TAMD_loss += self.cosine_loss(self.projector[i - 1](hidden_states[i][:, 0, :]), teacher_embedings)

                TAMD_loss = TAMD_loss / n_layer_distill

                for i in range(1, self.n_layer):
                    h_embedding = self.get_embedding(hidden_states[i], attention_mask)
                    next_h_embedding = self.get_embedding(hidden_states[i+1], attention_mask)
                    LASD_loss += self.structure_loss(h_embedding, next_h_embedding)
                LASD_loss = LASD_loss / (self.n_layer - 1)

        return TAMD_loss, LASD_loss

    def compute_InfoNCE_loss(self, scores, target):
        return self.cross_entropy(scores, target)

    def get_embedding(self, hidden_state, attention_mask):
        return get_embedding(hidden_state, attention_mask)

    def compute_loss(self, student_inputs, teacher_embedings=None):
        inputs = {key: value.to(self.device) for key, value in student_inputs.items()}

        student_q_outputs = self.student(**inputs, return_dict=True)
        student_p_outputs = self.student(**inputs, return_dict=True)

        student_q_embeddings = self.get_embedding(student_q_outputs.last_hidden_state,
                                                  inputs['attention_mask'])
        student_p_embeddings = self.get_embedding(student_p_outputs.last_hidden_state,
                                                  inputs['attention_mask'])
        q_dense_vecs = nn.functional.normalize(student_q_embeddings, dim=-1)
        p_dense_vecs = nn.functional.normalize(student_p_embeddings, dim=-1)

        targets = torch.arange(q_dense_vecs.size(0), device=self.device, dtype=torch.long)

        # dense loss
        dense_scores = self.compute_scores(q_dense_vecs, p_dense_vecs, self.args.temperature)
        simcse_loss = self.compute_InfoNCE_loss(dense_scores, targets)


        TAMD_loss, LASD_loss = self.distill_loss(student_q_outputs,
                                                 teacher_embedings,
                                                 inputs['attention_mask'])


        loss = 0.001 * simcse_loss + 0.75 * TAMD_loss + LASD_loss


        self.step += 1

        return loss, simcse_loss


In [ ]:
from torch import optim
# from muon import Muon
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
from transformers import get_scheduler

In [ ]:
eval_clss_tasks = [('data/multitask-data/multi-data/banking_train.csv',
                   'data/multitask-data/multi-data/banking77_validation.csv'),
                  ('data/multitask-data/multi-data/emotion_train.csv',
                   'data/multitask-data/multi-data/emotion_validation.csv'),
                  ('data/multitask-data/multi-data/tweet_train.csv',
                   'data/multitask-data/multi-data/tweet_validation.csv')]

eval_sts_tasks = ['data/multitask-data/multi-data/sick_validation.csv',
                  'data/multitask-data/multi-data/sts12_validation.csv',
                  'data/multitask-data/multi-data/stsb_validation.csv']

eval_pair_tasks = ['data/multitask-data/multi-data/mrpc_validation.csv',
                   'data/multitask-data/multi-data/scitail_validation.csv',
                   'data/multitask-data/multi-data/wic_validation.csv']

test_clss_tasks = [('data/multitask-data/multi-data/banking_train.csv',
                   'data/multitask-data/multi-data/banking77_test.csv'),
                  ('data/multitask-data/multi-data/emotion_train.csv',
                   'data/multitask-data/multi-data/emotion_test.csv'),
                  ('data/multitask-data/multi-data/tweet_train.csv',
                   'data/multitask-data/multi-data/tweet_test.csv')]

test_sts_tasks = ['data/multitask-data/multi-data/sick_test.csv',
                  'data/multitask-data/multi-data/sts12_test.csv',
                  'data/multitask-data/multi-data/stsb_test.csv']

test_pair_tasks = ['data/multitask-data/multi-data/mrpc_test.csv',
                   'data/multitask-data/multi-data/scitail_test.csv',
                   'data/multitask-data/multi-data/wic_test.csv']

In [ ]:
model = AutoModel.from_pretrained(
    args.student_model,
    device_map=args.student_device,
    output_hidden_states=True,
    attn_implementation="eager",
)

proj_hidden_layers = nn.ModuleList()

for _ in range(model.config.num_hidden_layers):
    proj = nn.Linear(model.config.hidden_size, args.teacher_embedding_dimension, bias=False)
    with torch.no_grad():
        proj.weight.normal_(mean=0.0, std=1e-3)
    proj_hidden_layers.append(proj)

proj_hidden_layers = proj_hidden_layers.to(model.device)

trainer = Trainer(model, proj_hidden_layers, args)

In [ ]:
muon_params = []
adamw_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue

    is_edge_or_head = any(keyword in name
                          for keyword in ["embeddings", "pooler", "projector",
                                          "classifier", "head", "layer.0",
                                          "layer.1", "layer.2", "layer.3", "layer.11"])

    is_1d_or_norm = param.ndim < 2 or "LayerNorm" in name or "bias" in name

    if not is_edge_or_head and not is_1d_or_norm:
        muon_params.append(param)
    else:
        adamw_params.append(param)

    # adamw_params.append(param)



opt_muon = optim.Muon(muon_params, lr=0.001, momentum=0.95)

opt_adamw = optim.AdamW(adamw_params, lr=args.learning_rate)
opt_adamw.add_param_group({"params": proj_hidden_layers.parameters(),
                           "lr": 1e-4, "weight_decay": 0.01})


In [ ]:
from torch.autograd import grad


def _trainable_params(trainer, include_projector=False):
    """
    Parameters used for the gradient-covariance eigenvalue analysis.

    The BERT pooler is excluded because the current embedding loss does not
    depend on pooler_output.
    """
    params = [
        p
        for name, p in trainer.student.named_parameters()
        if p.requires_grad and not name.startswith("pooler.")
    ]

    if include_projector:
        params += [
            p for p in trainer.projector.parameters()
            if p.requires_grad
        ]

    return params


@torch.no_grad()
def _clone_batch_to_cpu(batch):
    """Keep exactly the same analysis batches for every epoch."""
    student_inputs, teacher_embeddings = batch

    student_inputs = {
        k: v.detach().cpu().clone()
        for k, v in student_inputs.items()
    }
    teacher_embeddings = teacher_embeddings.detach().cpu().clone()

    return student_inputs, teacher_embeddings


def get_fixed_eigen_batches(dataloader, num_batches=10):
    """
    Select fixed batches once before training so the per-epoch eigenvalues
    are comparable across epochs.
    """
    fixed_batches = []

    for batch_idx, batch in enumerate(dataloader):
        fixed_batches.append(_clone_batch_to_cpu(batch))

        if batch_idx + 1 >= num_batches:
            break

    if not fixed_batches:
        raise ValueError("train_loader is empty")

    return fixed_batches

def _compute_analysis_loss(
    trainer,
    batch,
    seed=42,
    loss_mode="simcse",
):
    student_inputs, teacher_embeddings = batch

    device = trainer.student.device
    student_inputs = {
        k: v.to(device) for k, v in student_inputs.items()
    }
    teacher_embeddings = teacher_embeddings.to(device)

    cuda_devices = []
    if device.type == "cuda":
        cuda_devices = [device.index]

    old_step = trainer.step

    with torch.random.fork_rng(devices=cuda_devices):
        torch.manual_seed(seed)
        if device.type == "cuda":
            torch.cuda.manual_seed(seed)

        total_loss, simcse_loss = trainer.compute_loss(
            student_inputs,
            teacher_embeddings,
        )

    trainer.step = old_step

    if loss_mode == "full":
        return total_loss
    elif loss_mode == "simcse":
        return simcse_loss
    else:
        raise ValueError(
            f"Unknown loss_mode={loss_mode!r}. Use 'full' or 'simcse'."
        )


def _hvp_fn(loss, params):
    first_grads = grad(loss, params, create_graph=True, allow_unused=True)
    first_grads = [
        g if g is not None else torch.zeros_like(p)
        for g, p in zip(first_grads, params)
    ]
    flat_grad = torch.cat([g.reshape(-1) for g in first_grads])

    def hvp(v):
        dot = (flat_grad * v).sum()
        second_grads = grad(dot, params, retain_graph=True, allow_unused=True)
        second_grads = [
            h if h is not None else torch.zeros_like(p)
            for h, p in zip(second_grads, params)
        ]
        return torch.cat([h.reshape(-1).detach() for h in second_grads])

    return hvp


def _subspace_power_iteration(hvp, dim, k, num_iters=20, device=None, seed=0):
    torch.manual_seed(seed)
    V = torch.randn(dim, k, device=device, dtype=torch.float32)
    V, _ = torch.linalg.qr(V)

    for _ in range(num_iters):
        AV = torch.stack([hvp(V[:, i]) for i in range(k)], dim=1)
        V, _ = torch.linalg.qr(AV)

    AV = torch.stack([hvp(V[:, i]) for i in range(k)], dim=1)
    T = V.t() @ AV                      # k x k Rayleigh-quotient matrix
    T = 0.5 * (T + T.t())               # ép đối xứng để khử sai số số học
    eigvals, _ = torch.linalg.eigh(T)
    return eigvals.flip(0)              # sắp giảm dần


def compute_hessian_eigenvalues(
    trainer,
    fixed_batches,
    top_k=5,
    seed=42,
    include_projector=False,
    loss_mode="full",
    num_power_iters=20,
):
    student_was_training = trainer.student.training
    projector_was_training = trainer.projector.training

    trainer.student.train()
    trainer.projector.train()

    params = _trainable_params(trainer, include_projector=include_projector)
    if not params:
        raise ValueError("No trainable parameters found for Hessian analysis.")

    dim = sum(p.numel() for p in params)
    device = trainer.student.device

    per_batch_topk = []

    try:
        for batch_idx, batch in enumerate(fixed_batches):
            trainer.student.zero_grad(set_to_none=True)
            trainer.projector.zero_grad(set_to_none=True)

            loss = _compute_analysis_loss(
                trainer, batch, seed=seed + batch_idx, loss_mode=loss_mode,
            )

            hvp = _hvp_fn(loss, params)

            topk = _subspace_power_iteration(
                hvp, dim=dim, k=top_k,
                num_iters=num_power_iters,
                device=device,
                seed=seed + batch_idx,
            )
            per_batch_topk.append(topk.detach().cpu())

            print(
                f"  Hessian batch {batch_idx + 1}/{len(fixed_batches)}: "
                f"loss={loss.item():.6f}, top_eig={topk[0].item():.6f}"
            )

            trainer.student.zero_grad(set_to_none=True)
            trainer.projector.zero_grad(set_to_none=True)
            del loss, hvp

        stacked = torch.stack(per_batch_topk, dim=0)   # [num_batches, top_k]
        top_eigenvalues = stacked.mean(dim=0)
        lambda_max = top_eigenvalues[0].item()

        print(f"Top {top_k} Hessian eigenvalues (avg qua batches): "
              f"{top_eigenvalues.numpy()}")
        print(f"lambda_max (Hessian) = {lambda_max:.6f}")

    finally:
        trainer.student.zero_grad(set_to_none=True)
        trainer.projector.zero_grad(set_to_none=True)
        trainer.student.train(student_was_training)
        trainer.projector.train(projector_was_training)

        if device.type == "cuda":
            torch.cuda.empty_cache()

    return lambda_max, top_eigenvalues

In [ ]:
num_steps = len(train_loader)
total_training_steps = num_steps * args.num_train_epochs

scheduler_adamw = get_scheduler(
    name='cosine_with_min_lr',
    optimizer=opt_adamw,
    num_warmup_steps=int(total_training_steps * args.warmup_ratio),
    num_training_steps=total_training_steps,
    scheduler_specific_kwargs={'min_lr': 5e-5}
)

scheduler_muon = get_scheduler(
    name='cosine_with_min_lr',
    optimizer=opt_muon,
    num_warmup_steps=int(total_training_steps * args.warmup_ratio),
    num_training_steps=total_training_steps,
    scheduler_specific_kwargs={'min_lr': 5e-6}
)

# ------------------------------------------------------------------
# Per-epoch eigenvalue analysis
# ------------------------------------------------------------------
EIGEN_NUM_BATCHES = 10
EIGEN_TOP_K = 3

# "simcse" = raw SimCSE/InfoNCE term (no 0.001 multiplier)
# "full"   = 0.001*SimCSE + 0.75*TAMD + LASD
EIGEN_LOSS_MODE = "full"

eigen_batches = get_fixed_eigen_batches(
    train_loader,
    num_batches=EIGEN_NUM_BATCHES,
)

eigenvalue_log = []
top_eigenvalues_log = []

best_result = 0


for epoch in range(args.num_train_epochs):
    print(('\n' + '%8s' + '%14s' + '%17s' * 2) % ('epoch', 'memory', 'loss', 'student_loss'))
    p_bar = tqdm(train_loader, total=len(train_loader))
    loss_total = 0
    student_loss_total = 0
    step = 0

    for batch in p_bar:
        student_inputs, teacher_embeddings = batch
        teacher_embeddings = teacher_embeddings.to(trainer.student.device)

        opt_muon.zero_grad(set_to_none=True)
        opt_adamw.zero_grad(set_to_none=True)

        with autocast(dtype=torch.bfloat16):
            loss, student_loss = trainer.compute_loss(
                student_inputs,
                teacher_embeddings,
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainer.student.parameters(),
            max_norm=1.0,
        )

        opt_muon.step()
        opt_adamw.step()

        scheduler_adamw.step()
        scheduler_muon.step()

        loss_total += loss.item()
        student_loss_total += student_loss.item()
        step += 1

        memory = f'{torch.cuda.memory_reserved() / 1E9:.4g}G'
        s = ('%8s' + '%14s' + '%17.5g' * 2) % (
            f'{epoch + 1}/{args.num_train_epochs}',
            memory,
            loss_total / step,
            student_loss_total / step,
        )
        p_bar.set_description(s)

    # ------------------------------------------------------------
    # Largest eigenvalue after this epoch
    # ------------------------------------------------------------
    if epoch in [0, 2, 4, 9, 29]:
        print(f"\nComputing Hessian eigenvalues after epoch {epoch + 1}...")

        lambda_max, top_eigenvalues = compute_hessian_eigenvalues(
            trainer=trainer,
            fixed_batches=eigen_batches,
            top_k=EIGEN_TOP_K,
            seed=42,
            include_projector=False,
            loss_mode=EIGEN_LOSS_MODE,
            num_power_iters=20,
        )

        eigenvalue_log.append(lambda_max)
        top_eigenvalues_log.append(top_eigenvalues.tolist())

        print(
            f"[Epoch {epoch + 1}/{args.num_train_epochs}] "
            f"lambda_max(Hessian) = {lambda_max:.6f}"
        )

In [ ]:
print(
    "Largest gradient-covariance eigenvalue per epoch:",
    eigenvalue_log,
)

print(
    "Top eigenvalues per epoch:",
    top_eigenvalues_log,
)
